Librarii

In [1]:
# Import required libraries
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm


D:\school\homework\mirpr\Breast-Cancer-App\software-solution\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Data Loading
* dataset1: 3 foldere normal/malign/benign, cu imagini
* dataset2: Isoric al pacientilor
* dataset3: Metadata

Clasa Dataset pt a lua datele

Am separat datele in malign sau benign/normal

In [9]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset


class MultiFolderMultimodalDataset(Dataset):

  def __init__(self, base_dir, transform=None):
    self.base_dir = base_dir
    self.transform = transform

    # Load tabular files from dataset2 and dataset3
    self.patient_df = pd.read_csv(
        os.path.join(base_dir, "dataset2", "patient_history_dataset.csv")
    )
    self.biomarker_df = pd.read_csv(
        os.path.join(base_dir, "dataset3", "molecular_biomarker_dataset.csv")
    )

    # Merge tabular datasets on common Patient ID key
    self.metadata = pd.merge(
        self.patient_df, self.biomarker_df, on="Patient ID", how="inner"
    )

    # Gather images and track which folder they came from
    self.samples = []
    classes = ["normal", "benign", "malignant"]
    for cls in classes:
      img_folder = os.path.join(base_dir, "dataset1", cls, "images")
      if os.path.exists(img_folder):
        for img_name in os.listdir(img_folder):
          if img_name.lower().endswith((".png", ".jpg", ".jpeg", ".tif")):
            self.samples.append({
                "img_path": os.path.join(img_folder, img_name),
                "class_name": cls,  # Keep track of folder label
            })

    self.samples_df = pd.DataFrame(self.samples)

  def __len__(self):
    return len(self.samples_df)

  def __getitem__(self, idx):
    row = self.samples_df.iloc[idx]

    # Load and transform image
    image = Image.open(row["img_path"]).convert("RGB")
    if self.transform:
      image = self.transform(image)

    # Extract numerical features using exact CSV column names
    tabular_tensor = torch.tensor(
        [
            float(self.metadata.iloc[idx].get("Age at Diagnosis", 0.0)),
            float(self.metadata.iloc[idx].get("Neoplasm Histologic Grade", 0.0)),
        ],
        dtype=torch.float32,
    )

    # OPTION A: Binary Mapping (Malignant = 1.0, Normal/Benign = 0.0)
    label_val = 1.0 if row["class_name"] == "malignant" else 0.0
    label = torch.tensor(label_val, dtype=torch.float32)

    return image, tabular_tensor, label

2. Multi-Modal Fusion
* Image Branch: Foloseste EfficientNet preantrenat (timm) pentru a extrage embeddings
* Tabular Branch: Foloseste un Multi-Layer Perceptron (MLP) pentru a procesa istoric si biomarker data
* Fusion Layer: Concateneaza embeddingurile

In [10]:
import timm
import torch
import torch.nn as nn


class CrossAttentionFusionModel(nn.Module):

  def __init__(
      self,
      model_name="efficientnet_b0",
      num_tabular_features=4,
      embed_dim=128,
      num_heads=4,
  ):
    super(CrossAttentionFusionModel, self).__init__()

    # 1. Image Encoder Backbone
    self.image_encoder = timm.create_model(
        model_name, pretrained=True, num_classes=0
    )
    img_feature_dim = self.image_encoder.num_features

    # Project image features to shared embedding space (embed_dim)
    self.img_projection = nn.Linear(img_feature_dim, embed_dim)

    # 2. Tabular Encoder MLP
    self.tabular_encoder = nn.Sequential(
        nn.Linear(num_tabular_features, 64),
        nn.ReLU(),
        nn.Linear(64, embed_dim),  # Project tabular features to embed_dim
        nn.ReLU(),
    )

    # 3. Cross-Attention Module
    # Key, Query, Value dimension = embed_dim
    self.cross_attention = nn.MultiheadAttention(
        embed_dim=embed_dim, num_heads=num_heads, batch_first=True
    )
    self.layer_norm = nn.LayerNorm(embed_dim)

    # 4. Final Classifier Head
    # Input is the attended representation (embed_dim * 2)
    self.classifier = nn.Sequential(
        nn.Linear(embed_dim * 2, 64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 1),  # Binary Logit output
    )

  def forward(self, images, tabular):
    # Extract & project image features -> Shape: [batch_size, embed_dim]
    img_feats = self.img_projection(self.image_encoder(images))

    # Extract & project tabular features -> Shape: [batch_size, embed_dim]
    tab_feats = self.tabular_encoder(tabular)

    # Reshape for Attention: [batch_size, seq_len=1, embed_dim]
    img_query = img_feats.unsqueeze(1)
    tab_key_value = tab_feats.unsqueeze(1)

    # Cross-Attention: Image queries Tabular features
    # attn_output represents image features weighted by clinical tabular context
    attn_output, _ = self.cross_attention(
        query=img_query, key=tab_key_value, value=tab_key_value
    )
    attn_output = self.layer_norm(
        img_query + attn_output
    ).squeeze(1)  # Residual connection

    # Concatenate the attended image vector with the original tabular vector
    fused_representation = torch.cat((attn_output, tab_feats), dim=1)

    # Output prediction
    logits = self.classifier(fused_representation)
    return logits.squeeze(1)

3. Separare Test and Train Data

In [11]:
from torch.utils.data import DataLoader, random_split
from torchvision import transforms

# Transformari
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

base_dir = "../data/dataset/"
full_dataset = MultiFolderMultimodalDataset(base_dir=base_dir, transform=transform)

#  80% train, 20% test
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size]
)

print(f"Total samples: {len(full_dataset)}")
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

Total samples: 780
Training samples: 624
Test samples: 156


4. Training and Test

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

#
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CrossAttentionFusionModel(
    model_name="efficientnet_b0", num_tabular_features=2
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 3

for epoch in range(epochs):
  print(f"\n--- Epoch {epoch+1}/{epochs} ---")

  # Training
  model.train()
  running_loss = 0.0

  for images, tabular, labels in train_loader:
    images = images.to(device)
    tabular = tabular.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images, tabular)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

  epoch_train_loss = running_loss / len(train_loader.dataset)
  print(f"Train Loss: {epoch_train_loss:.4f}")

  # Testing
  model.eval()
  correct = 0
  total = 0
  test_loss = 0.0

  with torch.no_grad():
    for images, tabular, labels in test_loader:
      images = images.to(device)
      tabular = tabular.to(device)
      labels = labels.to(device)

      outputs = model(images, tabular)
      loss = criterion(outputs, labels)
      test_loss += loss.item() * images.size(0)


      probs = torch.sigmoid(outputs)
      predicted = (probs >= 0.5).float()

      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  epoch_test_loss = test_loss / len(test_loader.dataset)
  test_accuracy = (correct / total) * 100 if total > 0 else 0

  print(
      f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_accuracy:.2f}%"
  )


--- Epoch 1/3 ---
Train Loss: 0.6084
Test Loss: 0.5464 | Test Accuracy: 73.08%

--- Epoch 2/3 ---
Train Loss: 0.4450
Test Loss: 0.3451 | Test Accuracy: 88.46%

--- Epoch 3/3 ---
Train Loss: 0.1739
Test Loss: 0.2777 | Test Accuracy: 91.03%


with 2 features extracted:
Age at Diagnosis |
Neoplasm Histologic Grade

--- Epoch 1/3 ---
Train Loss: 0.5776
Test Loss: 0.4771 | Test Accuracy: 75.64%

--- Epoch 2/3 ---
Train Loss: 0.2987
Test Loss: 0.3309 | Test Accuracy: 85.90%

--- Epoch 3/3 ---
Train Loss: 0.1259
Test Loss: 0.2525 | Test Accuracy: 91.03%


with 8 features extracted:
Age at Diagnosis |
Tumor Size |
Lymph nodes examined positive |
Nottingham prognostic index |
Mutation Count |
Neoplasm Histologic Grade |
Tumor Stage |
ER Status

--- Epoch 1/3 ---
Train Loss: 0.5424
Test Loss: 0.4770 | Test Accuracy: 76.28%

--- Epoch 2/3 ---
Train Loss: 0.2789
Test Loss: 0.3461 | Test Accuracy: 85.90%

--- Epoch 3/3 ---
Train Loss: 0.1350
Test Loss: 0.3247 | Test Accuracy: 83.33%




--- Epoch 1/3 ---
Train Loss: 0.5918
Test Loss: 0.5285 | Test Accuracy: 75.00%

--- Epoch 2/3 ---
Train Loss: 0.4141
Test Loss: 0.3580 | Test Accuracy: 85.90%

--- Epoch 3/3 ---
Train Loss: 0.1641
Test Loss: 0.2997 | Test Accuracy: 89.74%
